# ToysAD-8K Official Evaluation Release

**Notebook file:** `10r-toysad8k-official-release.ipynb`

This trusted notebook prepares one official split at a time. Run it once for seen-test and once for unseen-test. Across the two runs it writes four physically separate resources: seen model-facing RGB, unseen model-facing RGB, seen evaluator-only records, and unseen evaluator-only records. Each run verifies the released master, the source archive, official split membership, and every extracted RGB member.

Model-facing manifests use opaque identifiers and an exact field allowlist. Geometry, labels, source identifiers, categories, and the private crosswalk remain in evaluator-only resources. They are never passed to a model notebook.

## Release contract

- Seen-test contains 715 scenes, 113 asset groups, and 14,300 RGB images.
- Unseen-test contains 2,437 scenes, 316 asset groups, and 48,740 RGB images.
- The minimal Notebook 09D receipt must verify protocol completion without retuning before the source archive is opened. This is not a scientific performance acceptance decision.
- Source membership and opaque model membership are verified with separate SHA-256 values.
- Existing files are reused only when their saved content hashes and the complete input signature match.
- Incremental versions process a bounded number of new scenes and publish a deterministic recovery ZIP only after CRC and SHA-256 verification.
- A later version attaches exactly one recovery output through `VPI_RESUME_ROOT`; role, source, membership, and release identities must match.
- Final model and evaluator package records are not written before exact split coverage.
- Safe evidence is aggregate only. ClearML receives no evaluator records, source identifiers, or RGB files.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "10r-toysad8k-official-release.ipynb"
RELEASE_ROLE = os.getenv("VPI_OFFICIAL_RELEASE_ROLE", "seen").strip().lower()
if RELEASE_ROLE not in {"seen", "unseen"}:
    raise ValueError("VPI_OFFICIAL_RELEASE_ROLE must be seen or unseen")
RUN_PHASE = os.getenv("VPI_RUN_PHASE", "incremental").strip().lower()
if RUN_PHASE not in {"incremental", "full"}:
    raise ValueError("VPI_RUN_PHASE must be incremental or full")
MAXIMUM_NEW_SCENES = int(os.getenv("VPI_MAXIMUM_NEW_SCENES", "100"))
if MAXIMUM_NEW_SCENES < 1:
    raise ValueError("VPI_MAXIMUM_NEW_SCENES must be positive")
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    f"/kaggle/working/toysad8k_official_{RELEASE_ROLE}_release" if EXECUTION_ENVIRONMENT == "kaggle" else f".local/runs/toysad8k_official_{RELEASE_ROLE}_release",
)).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RECOVERY_OUTPUT_ROOT = Path(os.getenv(
    "VPI_RECOVERY_OUTPUT_ROOT",
    f"/kaggle/working/official_{RELEASE_ROLE}_release_recovery"
    if EXECUTION_ENVIRONMENT == "kaggle"
    else f".local/runs/official_{RELEASE_ROLE}_release_recovery",
)).expanduser().resolve()
RECOVERY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:

def recovery_archives(root: Path, filename: str) -> list[Path]:
    """Find a ZIP or byte-preserved Kaggle upload; the caller verifies its hashes."""
    return sorted({path.resolve() for name in (filename, filename + ".bin")
                   for path in root.rglob(name) if path.is_file()}, key=lambda path: path.as_posix())

def kaggle_roots() -> list[Path]:
    """Return dataset and notebook mounts without their container directories."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    candidates = [path for path in root.iterdir()
                  if path.is_dir() and path.name not in {"datasets", "notebooks", "models"}]
    for pattern in ("datasets/*/*", "notebooks/*/*", "models/*/*/*/*"):
        for path in root.glob(pattern):
            if not path.is_dir():
                continue
            versions = path / "versions"
            if versions.is_dir():
                candidates.extend(child for child in versions.iterdir() if child.is_dir())
            else:
                candidates.append(path)
    return sorted({path.resolve() for path in candidates}, key=lambda path: path.as_posix())


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        if (candidate / "scripts/run_official_inference.py").is_file():
            return candidate.resolve()
        if (candidate / "code/scripts/run_official_inference.py").is_file():
            return (candidate / "code").resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, environment_name: str) -> Path:
    configured = os.getenv(environment_name)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_release_recovery_pair(role: str) -> tuple[Path, Path] | None:
    configured = os.getenv("VPI_RESUME_ROOT")
    sources = [Path(configured).expanduser()] if configured else kaggle_roots()
    bundles = sorted({
        path.resolve()
        for source in sources
        for path in recovery_archives(source, f"official-{role}-release-recovery.zip")
        if path.is_file()
    })
    manifests = sorted({
        path.resolve()
        for source in sources
        for path in source.rglob(f"official-{role}-release-recovery-manifest.json")
        if path.is_file()
    })
    if not bundles and not manifests:
        return None
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one official release recovery pair")
    return bundles[0], manifests[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source_root = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source_root if not existing else source_root + os.pathsep + existing
    return environment


def read_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)


def enable_tracking(environment: dict[str, str]) -> str:
    access = read_secret("CLEARML_API_ACCESS_KEY")
    secret = read_secret("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    return "online"


def prepare_tracking_runtime(mode: str, environment: dict[str, str]) -> str:
    if mode != "online":
        return mode
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML dependency setup was unavailable: {type(error).__name__}")
        return "disabled"
    return mode

## Verify the internal-lock release decision

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/official-evaluation/10r-toysad8k-official-release.ipynb",
)

LOCK_RESULT_RECEIPT = find_named_file(
    "internal_lock_result_receipt.json",
    "VPI_LOCK_RESULT_RECEIPT",
)
from visual_peer_inspector.workflows.official_release import verify_internal_lock_decision

lock_result = verify_internal_lock_decision(LOCK_RESULT_RECEIPT)
display({
    "internal-lock decision": lock_result["status"],
    "decision SHA-256": lock_result["sha256"],
    "source package opened": False,
})

## Locate the pinned source package

In [ ]:
configured = os.getenv("VPI_SOURCE_ROOT")
source_candidates = [Path(configured).expanduser()] if configured else kaggle_roots()
SOURCE_ROOT = next(
    candidate.resolve()
    for candidate in source_candidates
    if (candidate / "archive_index.sqlite3").is_file()
    and (candidate / "archive_index.sqlite3").is_file()
    and (candidate / "toysad8k_scene_master.json").is_file()
    and (candidate / "package-checksums.sha256").is_file()
)
MASTER = SOURCE_ROOT / "toysad8k_scene_master.json"
display({"source package located": True, "output root": str(OUTPUT_ROOT)})

## Build the four separated resources

The split run checks the full archive SHA-256. Every required member is also checked against the archive index, its local ZIP header, size, and CRC. The indexed RGB size, restored state, requested increment, and free working storage are checked before extraction. Set `VPI_OFFICIAL_RELEASE_ROLE` to `seen` or `unseen`; the default is `seen`. Use incremental versions until the complete package can be finalised, then publish the model input and evaluator output as different private datasets.

In [ ]:
environment = code_environment()
tracking_mode = prepare_tracking_runtime(enable_tracking(environment), environment)
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_official_release.py"),
    "--source-root", str(SOURCE_ROOT),
    "--master", str(MASTER),
    "--output-root", str(OUTPUT_ROOT),
    "--checkpoint-every", "10",
    "--clearml-mode", tracking_mode,
    "--role", RELEASE_ROLE,
    "--lock-result-receipt", str(LOCK_RESULT_RECEIPT),
    "--run-mode", RUN_PHASE,
    "--maximum-new-scenes", str(MAXIMUM_NEW_SCENES),
    "--recovery-output-directory", str(RECOVERY_OUTPUT_ROOT),
]
recovery_pair = find_release_recovery_pair(RELEASE_ROLE)
if recovery_pair is not None:
    command.extend(["--recovery-bundle", str(recovery_pair[0]), "--recovery-manifest", str(recovery_pair[1])])
subprocess.run(command, check=True, env=environment)

## Verify release evidence

In [ ]:
from visual_peer_inspector.data.cohort_contract import OFFICIAL_SEEN_TEST_CONTRACT, OFFICIAL_UNSEEN_TEST_CONTRACT
from visual_peer_inspector.data.model_safe_package import verify_model_input_package

contract = OFFICIAL_SEEN_TEST_CONTRACT if RELEASE_ROLE == "seen" else OFFICIAL_UNSEEN_TEST_CONTRACT
root = OUTPUT_ROOT / f"{RELEASE_ROLE}-model-input"
evaluator_root = OUTPUT_ROOT / f"{RELEASE_ROLE}-evaluator-only"
run_record = json.loads((OUTPUT_ROOT / "safe/run_record.json").read_text(encoding="utf-8"))
RELEASE_COMPLETE = run_record["status"] == "completed"
if RELEASE_COMPLETE:
    audit = verify_model_input_package(root, contract=contract)
    evaluator_manifest = json.loads((evaluator_root / "evaluator_manifest.json").read_text(encoding="utf-8"))
    verified = [{
        "cohort": contract.role,
        "scenes": audit.scene_count,
        "images": audit.image_count,
        "model manifest SHA-256": audit.manifest_sha256,
        "evaluator files": len(evaluator_manifest["files"]),
        "evaluator is model input": False,
        "status": "complete",
    }]
else:
    recovery_manifest = next(RECOVERY_OUTPUT_ROOT.glob("*-recovery-manifest.json"))
    verified = [{
        "cohort": contract.role,
        "scenes": run_record["completed_scene_count"],
        "images": run_record["completed_scene_count"] * contract.views_per_scene,
        "model manifest SHA-256": None,
        "evaluator files": 0,
        "evaluator is model input": False,
        "status": "checkpointed",
        "recovery manifest": str(recovery_manifest),
    }]
display(pd.DataFrame(verified))
display(NotebookImage(filename=str(OUTPUT_ROOT / "safe/official_cohort_summary.png")))

## Publication boundary

For a partial run, publish only the recovery ZIP and sidecar as one private recovery output and attach it to the next version. For a complete seen run, publish `seen-model-input` and `seen-evaluator-only` as two different private datasets. For a complete unseen run, publish `unseen-model-input` and `unseen-evaluator-only` the same way. Do not attach an evaluator dataset to Notebook 10A. Keep `trusted-release-state` as restart evidence outside model inputs. The `safe` directory and bounded ClearML task contain aggregate evidence only. If a split cannot fit the verified working volume, use a larger private output volume or a direct private dataset version upload; do not start extraction after the storage preflight fails.